# Custom Data Preprocessing for JPF-WCA Dataset

This script demonstrates how we perform our own data transformations for our own custom dataset. We implement `make_map_fn` functions to extract answers and format each example according to the required structure. The steps include:

- Loading the dataset that we created manually.
- Processing each example using a custom mapping function:
    - Constructing a data item with the fields: `data_source`, `prompt`, `ability`, `reward_model`, and `extra_info`.
- Saving the processed dataset in parquet format locally.
- Copying the local data to HDFS.

You can modify these functions to suit your own dataset or task requirements.


In [1]:
import pandas as pd

df = pd.read_parquet("eval-instruct.parquet")


In [2]:
df

,problem,problem_uuid,example_indices,examples,question,answer_index,answer_constants,answer_solution,variable_mapping
0,SimpleAscendingLast,65781f44-555c-437e-8454-99674f68c978,"[1, 2, 3]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,26,(declare-const c_13 Int)\n(declare-const in_16...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'k_11', 'i..."
1,SimpleAscendingLast,ca4a2e6b-85d4-43cf-9569-e9629aecc767,"[1, 2, 4]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,14,(declare-const o_9 Int)\n(declare-const j_8 In...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'o_9', 'in..."
2,SimpleAscendingLast,38643ce7-d3aa-4299-8136-319b5f40bd46,"[1, 2, 5]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,11,(declare-const m_1 Int)\n(declare-const global...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'global_8'..."
3,SimpleAscendingLast,7c9f6a71-9b44-4790-9dd2-2ef37c6b05d8,"[1, 2, 6]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,21,(declare-const r_10 Int)\n(declare-const local...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'r_10', 'i..."
4,SimpleAscendingLast,1b9f7464-d89c-49a8-b14b-d144be3df358,"[1, 2, 7]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,20,(declare-const k_10 Int)\n(declare-const t_9 I...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'k_10', 'i..."
...,...,...,...,...,...,...,...,...,...
21291,ComplexHalfEqual,0ac80b55-9513-49b4-8003-62239f0da748,"[1, 2, 3, 5, 6, 7, 8, 9, 10]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,22,(declare-const m_10 Int)\n(declare-const g_9 I...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'm_10', 'i..."
21292,ComplexHalfEqual,29bf299d-271d-47a0-9bd0-20fd0f442e99,"[1, 2, 4, 5, 6, 7, 8, 9, 10]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,18,(declare-const v_13 Int)\n(declare-const param...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'v_13', 'i..."
21293,ComplexHalfEqual,eeb1c5c5-d771-4efd-8d13-3bf3dc6d41fb,"[1, 3, 4, 5, 6, 7, 8, 9, 10]","[{'index': 1, 'solution': None}, {'index': 3, ...",<|im_start|>system\n You are a helpful assi...,13,(declare-const u_9 Int)\n(declare-const o_8 In...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'u_9', 'in..."
21294,ComplexHalfEqual,dcfd1826-7839-4203-b6e6-7313916293d4,"[2, 3, 4, 5, 6, 7, 8, 9, 10]","[{'index': 2, 'solution': '(assert ( = r_9 u...",<|im_start|>system\n You are a helpful assi...,12,(declare-const h_10 Int)\n(declare-const var_2...,(assert (and (and (and (and (and (and (and (an...,"{'get0': None, 'get1': None, 'in0': 'r_9', 'in..."


In [3]:
df.loc[0]

problem                                           SimpleAscendingLast
problem_uuid                     65781f44-555c-437e-8454-99674f68c978
example_indices                                             [1, 2, 3]
examples            [{'index': 1, 'solution': None}, {'index': 2, ...
question            <|im_start|>system\n    You are a helpful assi...
answer_index                                                       26
answer_constants    (declare-const c_13 Int)\n(declare-const in_16...
answer_solution     (assert (and (and (and (and (and (and (and (an...
variable_mapping    {'get0': None, 'get1': None, 'in0': 'k_11', 'i...
Name: 0, dtype: object

In [4]:
# Check if there are any duplicates in problem_uuid
df["problem_uuid"].duplicated().any()

False

In [5]:
# Check if any row the examples array's solution is all None
df["examples"].apply(lambda x: all([e["solution"] is None for e in x])).any()

True

In [6]:
# Show all the rows that have examples with all None solutions
df[df["examples"].apply(lambda x: all([e["solution"] is None for e in x]))]

,problem,problem_uuid,example_indices,examples,question,answer_index,answer_constants,answer_solution,variable_mapping
13552,SimpleEveryThird,40347c5d-ca96-4113-8e78-63dbc904df61,"[1, 2, 3]","[{'index': 1, 'solution': None}, {'index': 2, ...",<|im_start|>system\n You are a helpful assi...,22,(declare-const p_4 Int)\n(declare-const i_1 In...,(assert (and (and (and (and (and (and ( = y_...,"{'get0': None, 'get1': None, 'in0': None, 'in0..."


In [7]:
df.drop(df[df["examples"].apply(lambda x: all([e["solution"] is None for e in x]))].index, inplace=True)

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")

# Remove all the question that are greater than 1.8k tokens
print("Before filtering", len(df))

df.drop(df[df["question"].apply(lambda x: len(tokenizer(x)["input_ids"])) > 1800].index, inplace=True)

print("After filtering", len(df))

/Users/dankoh/ConStruct-veRL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Before filtering 21295
After filtering 17286


In [11]:
1-(17353/21295)

0.1851138764968302

In [9]:
def make_map_fn(split):
    def process_fn(row):
        data = {
            "data_source": "dannkoh/ConStruct-Base",
            "prompt": [{"role": "user", "content": row["question"]}],
            "ability": "generalisation",
            "reward_model": {"style": "rule", "ground_truth": row["answer_solution"]},
            "extra_info": {
                "answer_constants": row["answer_constants"],
                "variable_mapping": row["variable_mapping"],
                "answer_index": row["answer_index"],
                "example_indices": row["example_indices"],
                "problem_uuid": row["problem_uuid"],
            },
        }
        return data

    return process_fn

In [10]:
# Split the data into 2 parts: one for training and one for testing (75% training, 25% testing)
train = df.sample(frac=0.75, random_state=0)
test = df.drop(train.index)

train_dataset = pd.DataFrame(train.apply(make_map_fn("train"), axis=1).tolist())
test_dataset = pd.DataFrame(test.apply(make_map_fn("test"), axis=1).tolist())

In [11]:
train_dataset.to_parquet("train-instruct.parquet")
test_dataset.to_parquet("test-instruct.parquet")

In [15]:
train_dataset

,data_source,prompt,ability,reward_model,extra_info
0,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert (an...",{'answer_constants': '(declare-const c_5 Int) ...
1,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert (an...",{'answer_constants': '(declare-const t_3 Int) ...
2,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert (an...",{'answer_constants': '(declare-const a_1 Int) ...
3,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert ( ...",{'answer_constants': '(declare-const var_1 Int...
4,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert (an...",{'answer_constants': '(declare-const out_7 Int...
...,...,...,...,...,...
13010,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert (an...",{'answer_constants': '(declare-const i_8 Int) ...
13011,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': None}","{'answer_constants': None, 'variable_mapping':..."
13012,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert (an...",{'answer_constants': '(declare-const var_2 Int...
13013,dannkoh/ConStruct-Base,"[{'role': 'user', 'content': 'A conversation b...",generalisation,"{'style': 'rule', 'ground_truth': '(assert (an...",{'answer_constants': '(declare-const var_3 Int...


In [12]:
# max, min, mean prompt token length of qwen tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")

print("Max prompt token length:", df["question"].apply(tokenizer.tokenize).apply(len).max())
print("Min prompt token length:", df["question"].apply(tokenizer.tokenize).apply(len).min())
print("Mean prompt token length:", df["question"].apply(tokenizer.tokenize).apply(len).mean())

Max prompt token length: 1800


KeyboardInterrupt: 

In [ ]:
temp = df.copy()
temp["token_length"] = df["question"].apply(tokenizer.tokenize).apply(len)

temp["token_length"].hist()

temp["token_length"].describe()

In [ ]:
# get tokenized prompt length distribution

distribution = df["question"].apply(tokenizer.tokenize).apply(len).value_counts().sort_index()

In [ ]:
# Plot distribution and take a subset of the data to use.
import matplotlib.pyplot as plt

plt.bar(distribution.index, distribution.values)
plt.xlabel("Tokenized prompt length")
plt.ylabel("Count")
plt.title("Tokenized prompt length distribution")
plt.show()

# Take the 90th percentile of the data
df["question"].apply(tokenizer.tokenize).apply(len).quantile(0.9)